# Clean-hard targeted training vs random control

RQ1: does up-weighting hard rows with a clear label beat up-weighting the same number of random rows? "Hard" is tested two ways: rows the baseline is **unsure** about (the 0.4-0.6 band) and rows it got **wrong** out-of-fold (Just Train Twice, Liu et al. 2021).
RQ2: does the answer change when the FinBERT embeddings are fine-tuned instead of frozen?

Every run retrains the FinBERT CNN from scratch with the baseline recipe; only the loss weights change.

| arm | rows weighted K (all others 1) |
|---|---|
| `no_weight` | none, the reference |
| `clean_hard` | 103 hard + all-agree rows |
| `random` | 103 random train rows (new draw per seed) |
| `clean_random` | 103 all-agree, non-hard rows (new draw per seed) |
| `variant` | clean-hard rows get K, the 261 hard rows without agreement get 0.5 |
| `clean_error` | 114 out-of-fold errors that are all-agree (Just Train Twice) |
| `error_random` | 114 random train rows (new draw per seed) |
| `error_clean_random` | 114 all-agree rows the baseline got right (new draw per seed) |

Row sets come from `results/rq1_rq2/row_sets.csv`, scores are on **val** only. K is chosen here; test is run once at the end.
Every run goes to `results/rq1_rq2/runs.csv` (re-running a run replaces it) and its val probabilities to `results/rq1_rq2/probs/` for the taxonomy.

## Imports

In [1]:
import os
os.environ["HF_HOME"] = os.path.abspath("../.cache/huggingface")
os.environ["HF_HUB_CACHE"] = os.path.abspath("../.cache/huggingface/hub")
import sys

import numpy as np
import pandas as pd

sys.path.append("..")
from utils.clean_hard import run_arm, save_runs
from utils.finbert_cnn import SEEDS, load_finbert_inputs
from utils.preprocessing import encode_labels

## Data

In [2]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
y_train = encode_labels(train_df.label)
y_val = encode_labels(val_df.label)
X_train, X_val, embedding_matrix = load_finbert_inputs(train_df, val_df)

row_sets = pd.read_csv("../results/rq1_rq2/row_sets.csv")
data = dict(embedding_matrix=embedding_matrix, X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
            band_val=pd.read_csv("../results/hard_samples/band_val.csv"))

RUNS_PATH = "../results/rq1_rq2/runs.csv"
PROBS_DIR = "../results/rq1_rq2/probs"
os.makedirs(PROBS_DIR, exist_ok=True)
KS = [3, 5, 10]
KS_ERROR = [5, 10, 20]   # Just Train Twice uses larger weights
print(row_sets.groupby("row_set").train_idx.count() // len(SEEDS))

/Users/cleojj002/Documents/GitHub/nlp-financial-phrasebank/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1106 16:53:05.160000 52018 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 36959.95it/s]
[transformers] BertModel LOAD REPORT from: yiyanghkust/finbert-pretrain
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | U

row_set
ambiguous_hard        261
clean_error           114
clean_hard            103
clean_random          103
error_clean_random    114
error_random          114
random                103
Name: train_idx, dtype: int64


In [3]:
def run_seeds(arm, k, trainable=False, epochs=20):
    rows = []
    for seed in SEEDS:
        row, probs = run_arm(arm, k, seed, data, row_sets, trainable=trainable, epochs=epochs)
        rows.append(row)
        np.save(f"{PROBS_DIR}/{arm}_k{row['k']}_{row['embedding_config']}_s{seed}.npy", probs)
        print(f"{arm} k={row['k']} seed={seed}: f1={row['f1_overall']:.3f}  hard={row['f1_hard']:.3f}")
    return save_runs(RUNS_PATH, pd.DataFrame(rows))

## 1. No-weight reference (run this first)

Check: every seed should land around 0.74-0.77. A seed at about 0.248 means it got stuck predicting neutral; stop and fix that before running the other arms.

In [4]:
runs = run_seeds("no_weight", 1)

 - val_macro_f1: 0.3184
 - val_macro_f1: 0.4151
 - val_macro_f1: 0.4368
 - val_macro_f1: 0.5897
 - val_macro_f1: 0.6485
 - val_macro_f1: 0.6796
 - val_macro_f1: 0.7090
 - val_macro_f1: 0.7188
 - val_macro_f1: 0.7110
 - val_macro_f1: 0.7303
 - val_macro_f1: 0.7102
 - val_macro_f1: 0.7318
 - val_macro_f1: 0.7275
 - val_macro_f1: 0.7436
 - val_macro_f1: 0.7451
 - val_macro_f1: 0.7605
 - val_macro_f1: 0.7653
 - val_macro_f1: 0.7455
 - val_macro_f1: 0.7468
 - val_macro_f1: 0.7354
no_weight k=1 seed=42: f1=0.765  hard=0.606
 - val_macro_f1: 0.3505
 - val_macro_f1: 0.4550
 - val_macro_f1: 0.6215
 - val_macro_f1: 0.6913
 - val_macro_f1: 0.7415
 - val_macro_f1: 0.7549
 - val_macro_f1: 0.7519
 - val_macro_f1: 0.7570
 - val_macro_f1: 0.7603
 - val_macro_f1: 0.7525
 - val_macro_f1: 0.7608
 - val_macro_f1: 0.7498
 - val_macro_f1: 0.7576
 - val_macro_f1: 0.7508
 - val_macro_f1: 0.7296
 - val_macro_f1: 0.7423
no_weight k=1 seed=43: f1=0.761  hard=0.617
 - val_macro_f1: 0.3246
 - val_macro_f1: 0.4144


## 2. Targeted arms (frozen embeddings)

In [ ]:
for k in KS:
    runs = run_seeds("clean_hard", k)

In [ ]:
for k in KS:
    runs = run_seeds("variant", k)

## 3. Controls (frozen embeddings)

In [ ]:
for k in KS:
    runs = run_seeds("random", k)

In [ ]:
for k in KS:
    runs = run_seeds("clean_random", k)

## 3b. Just Train Twice: up-weight the baseline's errors (frozen embeddings)

The targeted rows are the 114 train sentences the out-of-fold CNN got wrong and all annotators agreed on, so label noise is filtered out. Unlike the band, this includes confident errors (72 of the 114).

In [ ]:
for k in KS_ERROR:
    runs = run_seeds("clean_error", k)

In [ ]:
for k in KS_ERROR:
    runs = run_seeds("error_random", k)

In [ ]:
for k in KS_ERROR:
    runs = run_seeds("error_clean_random", k)

## 4. Compare on val and choose K

Mean and spread over the seeds, and the difference to the no-weight reference. Pick K (one for each definition of hard) where targeted minus its controls is largest **compared with the spread**; a difference smaller than the std is not an effect.

In [ ]:
runs = pd.read_csv(RUNS_PATH)
cols = ["f1_overall", "f1_hard", "f1_nonhard", "f1_allagree", "f1_not_allagree"]

summary = runs.groupby(["embedding_config", "arm", "k"])[cols].agg(["mean", "std"]).round(3)
summary

In [ ]:
frozen = runs[runs.embedding_config == "finbert_frozen"]
reference = frozen[frozen.arm == "no_weight"][cols].mean()
gain = frozen[frozen.arm != "no_weight"].groupby(["arm", "k"])[cols].mean() - reference
gain.round(3)

In [ ]:
# targeted minus each of its controls, per seed (same seeds, so the pairs line up)
pivot = frozen[frozen.arm != "no_weight"].pivot_table(index=["k", "seed"], columns="arm", values="f1_overall")
pairs = [("clean_hard", "random"), ("clean_hard", "clean_random"),
         ("clean_error", "error_random"), ("clean_error", "error_clean_random")]
for target, control in pairs:
    if {target, control} <= set(pivot.columns):
        diff = (pivot[target] - pivot[control]).dropna().groupby("k").agg(["mean", "std", "min", "max"])
        print(f"{target} - {control}\n{diff.round(3)}\n")


## 5. RQ2: fine-tuned embeddings at the chosen K

Run the targeted arm that did best in section 4 (clean-hard or clean-error) with its random control. Fine-tuned embeddings learn slowly at lr 5e-5, so they get 80 epochs.

In [ ]:
TARGET, CONTROL = "clean_hard", "random"   # or "clean_error", "error_random"
CHOSEN_K = None   # fill in after section 4
assert CHOSEN_K in (KS if TARGET == "clean_hard" else KS_ERROR), "choose K in section 4 first"

for arm, k in [("no_weight", 1), (TARGET, CHOSEN_K), (CONTROL, CHOSEN_K)]:
    runs = run_seeds(arm, k, trainable=True, epochs=80)
